In [ ]:
import os
import sys
import pickle
from pathlib import Path

from dotenv import load_dotenv
from neo4j import GraphDatabase

sys.path.insert(0, str(Path("..").resolve()))
load_dotenv("../.env")

In [ ]:
from qasa_rag.embedder import Embedder
from qasa_rag.retrieval import (
    QASARetriever,
    AblationConfig,
    AblationEvaluator,
)

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

DATASET_NAME = "2wiki" # "musique" or "2wiki"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
embedder = Embedder(cache_path=Path("cache/embeddings_cache.pkl"))

In [ ]:
with open(f"ground_truth-{DATASET_NAME}.pkl", "rb") as f:
    ground_truth = pickle.load(f)

print(f"Loaded {len(ground_truth)} questions for {DATASET_NAME}")
ground_truth[0]

## Ablation Configurations

Run configs one by one — each takes ~1h for 1000 questions. Use `n_eval` to test on a subset first.

In [ ]:
retriever_cls = QASARetriever

evaluator = AblationEvaluator(
    driver=driver,
    embedder=embedder,
    ground_truth=ground_truth,

    output_dir=f"ablation-results-{DATASET_NAME}",
    llm_model="gemini-2.5-flash",
    llm_judge=False,
    retriever_cls=retriever_cls,
    query_entity_cache_path=Path(f"cache/query_entities-{DATASET_NAME}.pkl"),
)

In [ ]:
configs = {
    # --- Baseline: no graph at all ---
    "naive_vector": AblationConfig(
        name="naive-vector",
        naive_vector=True,
        top_k_entities=30,
    ),

    # --- Ablation: remove query-awareness (uniform propagation) ---
    "no_qa": AblationConfig(
        name="no-query-aware",
        query_aware=False,
        max_steps=3,
        decay=0.7,
    ),

    # --- Vary max_steps (decay=0.7, query_aware=True) ---
    "steps_1": AblationConfig(name="steps-1", max_steps=1, decay=0.7),
    "steps_2": AblationConfig(name="steps-2", max_steps=2, decay=0.7),
    "steps_3": AblationConfig(name="steps-3", max_steps=3, decay=0.7),
    "steps_4": AblationConfig(name="steps-4", max_steps=4, decay=0.7),

    # --- Vary decay (max_steps=3, query_aware=True) ---
    "decay_03": AblationConfig(name="decay-0.3", max_steps=3, decay=0.3),
    "decay_05": AblationConfig(name="decay-0.5", max_steps=3, decay=0.5),
    "decay_07": AblationConfig(name="decay-0.7", max_steps=3, decay=0.7),
    "decay_09": AblationConfig(name="decay-0.9", max_steps=3, decay=0.9),
    "decay_10": AblationConfig(name="decay-1.0", max_steps=3, decay=1.0),
}

for c in configs.values():
    print(c.describe())

## Run Experiments

Run one config at a time. Set `n_eval` to a small number (e.g. 50) for a quick sanity check, then `None` for the full dataset.

In [ ]:
N_EVAL = 500  # set to None for full dataset

In [ ]:
evaluator.run_config(configs["naive_vector"], n_eval=N_EVAL, max_workers=10)

In [ ]:
evaluator.run_config(configs["no_qa"], n_eval=N_EVAL, max_workers=10)

In [ ]:
evaluator.run_config(configs["steps_1"], n_eval=N_EVAL, max_workers=10)

In [ ]:
evaluator.run_config(configs["steps_2"], n_eval=N_EVAL, max_workers=10)

In [ ]:
evaluator.run_config(configs["steps_3"], n_eval=N_EVAL, max_workers=10)

In [ ]:
evaluator.run_config(configs["steps_4"], n_eval=N_EVAL, max_workers=10)

In [ ]:
evaluator.run_config(configs["decay_03"], n_eval=N_EVAL, max_workers=10)

In [ ]:
evaluator.run_config(configs["decay_05"], n_eval=N_EVAL, max_workers=10)

In [ ]:
evaluator.run_config(configs["decay_07"], n_eval=N_EVAL, max_workers=10)

In [ ]:
evaluator.run_config(configs["decay_09"], n_eval=N_EVAL, max_workers=10)

In [ ]:
evaluator.run_config(configs["decay_10"], n_eval=N_EVAL, max_workers=10)

## Summary

In [ ]:
summary = evaluator.summary()
summary

In [ ]:
driver.close()